# 🐕 Module 2: Classical Feature-Based Vision

**50-Breed Subset | HOG + LBP + SIFT | SVM Classification**

In [ ]:
# ============ STEP 1: SETUP ============
!pip install opencv-python-headless scikit-image scikit-learn matplotlib tqdm seaborn -q

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import files, drive
from skimage.feature import hog, local_binary_pattern
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Step 1 Complete!")

In [ ]:
# ============ STEP 2: CHOOSE DATA SOURCE ============
USE_GOOGLE_DRIVE = True   # True = Google Drive, False = Manual upload

In [ ]:
# ============ STEP 2A: LOAD 50-BREED SUBSET FROM GOOGLE DRIVE ============
if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')
    DATASET_PATH = '/content/drive/MyDrive/CV-Project/images/Images'
    
    # === 50 BREEDS SUBSET (VERIFIED) ===
    SELECTED_50_BREEDS = [
        "Chihuahua", "Japanese_spaniel", "Maltese_dog", "Pomeranian", "toy_poodle",
        "Shih-Tzu", "papillon", "Yorkshire_terrier", "miniature_pinscher", "pug",
        "beagle", "basset", "cocker_spaniel", "Border_collie", "Shetland_sheepdog",
        "Australian_terrier", "Welsh_springer_spaniel", "English_springer", "Brittany_spaniel", "French_bulldog",
        "Boston_bull", "Staffordshire_bullterrier", "Pembroke", "Cardigan", "basenji",
        "golden_retriever", "Labrador_retriever", "German_shepherd", "Rottweiler", "Doberman",
        "boxer", "Great_Dane", "Saint_Bernard", "Siberian_husky", "malamute",
        "Samoyed", "collie", "Old_English_sheepdog", "Afghan_hound", "borzoi",
        "whippet", "chow", "Newfoundland", "Great_Pyrenees", "komondor",
        "Weimaraner", "vizsla", "Rhodesian_ridgeback", "bloodhound", "Irish_setter"
    ]
    
    # VERIFY: Exactly 50 breeds
    assert len(SELECTED_50_BREEDS) == 50, f"Expected 50, got {len(SELECTED_50_BREEDS)}"
    print(f"✅ Using 50-breed subset (verified: {len(SELECTED_50_BREEDS)} breeds)")
    
    images = []
    labels = []
    MAX_PER_BREED = 30
    
    folders = os.listdir(DATASET_PATH)
    breed_map = {f.split('-')[1]: f for f in folders if '-' in f}
    
    loaded_breeds = []
    for breed in tqdm(SELECTED_50_BREEDS, desc="Loading 50 breeds"):
        if breed not in breed_map:
            print(f"  ⚠️ Not found: {breed}")
            continue
        folder = os.path.join(DATASET_PATH, breed_map[breed])
        count = 0
        for f in os.listdir(folder)[:MAX_PER_BREED]:
            if f.endswith('.jpg'):
                img = cv2.imread(os.path.join(folder, f))
                if img is not None:
                    images.append(img)
                    labels.append(breed)
                    count += 1
        if count > 0:
            loaded_breeds.append(breed)
    
    print(f"\n✅ Loaded {len(images)} images from {len(loaded_breeds)}/50 breeds")

In [ ]:
# ============ STEP 2B: MANUAL UPLOAD (if not using Drive) ============
if not USE_GOOGLE_DRIVE:
    print("📤 Upload images: breedname_1.jpg, breedname_2.jpg")
    uploaded = files.upload()
    images, labels = [], []
    for filename, content in uploaded.items():
        breed = filename.rsplit('_', 1)[0].replace('-', '_').lower()
        img = cv2.imdecode(np.frombuffer(content, np.uint8), cv2.IMREAD_COLOR)
        if img is not None:
            images.append(img)
            labels.append(breed)
    print(f"✅ Loaded {len(images)} images")

In [ ]:
# ============ STEP 3: FEATURE VISUALIZATIONS ============
from skimage.feature import hog as hog_vis

sample_idx = [0, len(images)//3, 2*len(images)//3]

# --- HOG ---
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("HOG (Shape Descriptor)", fontsize=12)
for i, idx in enumerate(sample_idx):
    gray = cv2.cvtColor(images[idx], cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (128, 128))
    _, hog_img = hog_vis(resized, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), visualize=True)
    axes[i].imshow(hog_img, cmap='gray')
    axes[i].set_title(labels[idx][:12])
    axes[i].axis('off')
plt.tight_layout()
plt.show()

# --- SIFT ---
sift = cv2.SIFT_create(nfeatures=100)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("SIFT (Keypoints)", fontsize=12)
for i, idx in enumerate(sample_idx):
    gray = cv2.cvtColor(images[idx], cv2.COLOR_BGR2GRAY)
    kp, _ = sift.detectAndCompute(gray, None)
    img_kp = cv2.drawKeypoints(images[idx], kp, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    axes[i].imshow(cv2.cvtColor(img_kp, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"{labels[idx][:10]} ({len(kp)} kp)")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

# --- LBP ---
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("LBP (Texture Descriptor)", fontsize=12)
for i, idx in enumerate(sample_idx):
    gray = cv2.cvtColor(images[idx], cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (128, 128))
    lbp = local_binary_pattern(resized, P=24, R=3, method='uniform')
    axes[i].imshow(lbp, cmap='jet')
    axes[i].set_title(labels[idx][:12])
    axes[i].axis('off')
plt.tight_layout()
plt.show()

print("✅ Visualizations complete (showing 3 samples, processing all 50 breeds)")

In [ ]:
# ============ STEP 4: FEATURE EXTRACTION (ALL 50 BREEDS) ============

def extract_hog(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (128, 128))
    return hog(resized, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), feature_vector=True)

def extract_lbp(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (128, 128))
    lbp = local_binary_pattern(resized, 24, 3, method='uniform')
    h, w = resized.shape
    hist = []
    for r in [lbp[:h//2, :w//2], lbp[:h//2, w//2:], lbp[h//2:, :w//2], lbp[h//2:, w//2:]]:
        h_r, _ = np.histogram(r.ravel(), bins=26, range=(0, 26))
        hist.append(h_r.astype(np.float32) / (h_r.sum() + 1e-7))
    return np.concatenate(hist)

def extract_fused(image):
    h = extract_hog(image)
    return np.concatenate([h / (np.linalg.norm(h) + 1e-7), extract_lbp(image)])

print(f"Extracting features from {len(images)} images...")
X_hog = np.array([extract_hog(img) for img in tqdm(images, desc="HOG")])
X_lbp = np.array([extract_lbp(img) for img in tqdm(images, desc="LBP")])
X_fused = np.array([extract_fused(img) for img in tqdm(images, desc="Fused")])

le = LabelEncoder()
y = le.fit_transform(labels)

print(f"\n✅ Features: HOG={X_hog.shape[1]}, LBP={X_lbp.shape[1]}, Fused={X_fused.shape[1]}")
print(f"✅ Classes: {len(le.classes_)} breeds")

In [ ]:
# ============ STEP 5: PCA VISUALIZATION ============

X_scaled = StandardScaler().fit_transform(X_fused)
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 10))
for i, breed in enumerate(le.classes_[:20]):  # Show 20 for readability
    mask = y == i
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], label=breed[:12], alpha=0.6, s=30)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
plt.title(f"PCA: Feature Space (showing 20/{len(le.classes_)} breeds)")
plt.legend(bbox_to_anchor=(1.05, 1), fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# ============ STEP 6: ABLATION STUDY ============

def train_and_evaluate(X, y, name):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)
    svm = SVC(kernel='rbf', C=10, gamma='scale')
    svm.fit(X_tr, y_tr)
    y_pred = svm.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    return acc, y_te, y_pred, svm, scaler

print("\n" + "="*50)
print("ABLATION STUDY: Feature Comparison")
print("="*50)

acc_hog, _, _, _, _ = train_and_evaluate(X_hog, y, "HOG")
acc_lbp, _, _, _, _ = train_and_evaluate(X_lbp, y, "LBP")
acc_fused, y_test, y_pred, best_svm, best_scaler = train_and_evaluate(X_fused, y, "Fused")

print(f"\n{'Feature':<20} {'Accuracy':>10} {'vs Random':>12}")
print("-"*45)
random_baseline = 1/len(le.classes_)
print(f"{'HOG (shape)':<20} {acc_hog:>9.1%} {acc_hog/random_baseline:>10.1f}x better")
print(f"{'LBP (texture)':<20} {acc_lbp:>9.1%} {acc_lbp/random_baseline:>10.1f}x better")
print(f"{'HOG + LBP (fused)':<20} {acc_fused:>9.1%} {acc_fused/random_baseline:>10.1f}x better")
print("-"*45)
print(f"Random baseline: {random_baseline:.1%}")
print("="*50)

In [ ]:
# ============ STEP 7: CONFUSION MATRIX ============

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Get breed names for the test set
test_breeds = le.classes_

# Plot confusion matrix (show top 15 most confused)
plt.figure(figsize=(14, 12))

# Normalize for percentages
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Show only first 20 classes for readability
n_show = min(20, len(test_breeds))
cm_show = cm_norm[:n_show, :n_show]
breeds_show = [b[:10] for b in test_breeds[:n_show]]

sns.heatmap(cm_show, annot=True, fmt='.1%', cmap='Blues',
            xticklabels=breeds_show, yticklabels=breeds_show)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix (Normalized) - Top {n_show} breeds')
plt.tight_layout()
plt.show()

# Find most confused pairs
print("\n" + "="*50)
print("FAILURE ANALYSIS: Most Confused Breed Pairs")
print("="*50)

# Find off-diagonal max values
cm_copy = cm_norm.copy()
np.fill_diagonal(cm_copy, 0)

confused_pairs = []
for i in range(len(cm_copy)):
    for j in range(len(cm_copy)):
        if cm_copy[i, j] > 0.05:  # More than 5% confusion
            confused_pairs.append((test_breeds[i], test_breeds[j], cm_copy[i, j]))

confused_pairs.sort(key=lambda x: x[2], reverse=True)

print(f"\n{'True Breed':<20} {'Confused With':<20} {'Rate':>8}")
print("-"*50)
for true_b, pred_b, rate in confused_pairs[:10]:
    print(f"{true_b:<20} {pred_b:<20} {rate:>7.1%}")
    
if len(confused_pairs) == 0:
    print("No significant confusion pairs (all < 5%)")
print("="*50)

In [ ]:
# ============ STEP 8: SUMMARY ============

print(f"""
{'='*60}
🎓 MODULE 2 COMPLETE: Classical Feature-Based Vision
{'='*60}

DATASET:
  • Breeds: {len(le.classes_)} (50-breed subset)
  • Images: {len(images)} total
  • Train/Test: 80/20 split

ABLATION STUDY RESULTS:
  ┌──────────────────┬──────────┬─────────────┐
  │ Feature          │ Accuracy │ vs Random   │
  ├──────────────────┼──────────┼─────────────┤
  │ HOG (shape)      │ {acc_hog:>6.1%}   │ {acc_hog/random_baseline:>5.1f}x better │
  │ LBP (texture)    │ {acc_lbp:>6.1%}   │ {acc_lbp/random_baseline:>5.1f}x better │
  │ HOG+LBP (fused)  │ {acc_fused:>6.1%}   │ {acc_fused/random_baseline:>5.1f}x better │
  └──────────────────┴──────────┴─────────────┘
  Random baseline: {random_baseline:.1%}

KEY FINDINGS:
  • Fusion outperforms individual features
  • Classical methods struggle with fine-grained breeds
  • Similar breeds (e.g., Husky/Malamute) cause confusion

NEXT: Module 3 (Deep Learning) → Expected 70-85%
{'='*60}
""")